In [13]:
import requests
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import json
import os
import pandas as pd
import numpy as np
from pandas import DataFrame 
import re
from collections import defaultdict
import json

# Mongo 
uri = os.getenv('SBS_V1_MONGO_URI')

# rapidApi
headers = {
    "X-RapidAPI-Key": os.getenv('RAPID_API_KEY'),
    "X-RapidAPI-Host": os.getenv('RAPID_API_HOST')
}

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))
db = client["SBSV1"]
collection = db["nba_games_historical"]

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [14]:
nba_games_historical_collection = db["nba_games_historical"]
player_game_stats_historical_collection = db["player_game_stats_historical"]
player_game_stats_avgs_historical_collection = db["player_game_stats_avgs_historical"]

In [15]:
################## API FUNCTIONS ######################## 
#########################################################

#########################################################
# games by game ids #####################################
def get_team_nicknames_by_id():
    # team code by id
    url = "https://api-nba-v1.p.rapidapi.com/teams"
    response = requests.get(url, headers=headers).json()["response"]
    df = pd.DataFrame(response)
    df = df.loc[(df["nbaFranchise"] == True) & (df["allStar"] == False)]
    
    team_nickname_to_id_map = {}
    
    for index, row in df.iterrows():
        team_nickname_to_id_map.update({row["nickname"]: row["id"]})
    return team_nickname_to_id_map
#########################################################

#########################################################
# get player per team and season ########################  
def get_player_per_team_and_season(team, season):
    url = "https://api-nba-v1.p.rapidapi.com/players"
    querystring = {"team": team,"season": season }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    
    return df
#########################################################

#########################################################
# games by game ids #####################################
def get_games_by_game_ids(season, team_id):
    url = "https://api-nba-v1.p.rapidapi.com/games"
    querystring = {"season":season,"team":team_id}

    response = requests.get(url, headers=headers, params=querystring)
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    df = df.loc[(df["status.long"] == "Finished")]
    df = df.sort_values(by=["date.start"])
    
    return drop_cols(df, cols_to_drop_for_game_stats)
#########################################################

#########################################################
# insert player stats for each game #####################
def get_players_per_game_df(game_id):    
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"

    querystring = {"game": game_id }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    return drop_cols(df, cols_to_drop_for_player_stats)
#########################################################
    
#########################################################

In [16]:
################## util functions #######################
#########################################################

#########################################################
# Function to convert dot-separated to camelCase ########
def to_camel_case(s):
    parts = s.split('.')
    return parts[0] + ''.join(word.capitalize() for word in parts[1:])
#########################################################

#########################################################
# Function to recursively rename fields in a document and remove fields with periods
def rename_and_remove_fields(doc):
    if isinstance(doc, dict):
        new_doc = {}
        for key, value in doc.items():
            if '.' in key:
                new_key = to_camel_case(key)
                if isinstance(value, (dict, list)):
                    value = rename_and_remove_fields(value)
                new_doc[new_key] = value
            else:
                if isinstance(value, (dict, list)):
                    value = rename_and_remove_fields(value)
                new_doc[key] = value
        return new_doc
    elif isinstance(doc, list):
        return [rename_and_remove_fields(item) for item in doc]
    return doc
#########################################################

#########################################################
# drop all cols from a df ###############################
def drop_cols(df, cols):
    for col in cols:
        df = df.drop(col, axis=1)
    return df
#########################################################

#########################################################
# get dot separated keys ################################
def get_dot_separated_keys(document):
    dot_keys = [key for key in document if '.' in key]
    return dot_keys
#########################################################

#########################################################
# get nested dict #######################################
def nest_dict(flat_dict):
    nested_dict = {}
    for key, value in flat_dict.items():
        parts = key.split('.')
        d = nested_dict
        for part in parts[:-1]:
            if part not in d:
                d[part] = {}
            d = d[part]
        d[parts[-1]] = value
    return nested_dict
# Example
# nested_json = [nest_dict(record) for record in df.to_dict(orient='records')]
#########################################################

#########################################################
## zero non numeric values #############################
def zero_non_numeric_values(df):
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df
#########################################################

#########################################################
# rename player stats cols ##############################
def rename_col(col, new_prefix):
    l = col.split('.')
    if (len(l) > 1):
        return f"{new_prefix}.{l[1]}"
    return new_prefix
#########################################################

#########################################################
# remove prefix #########################################
def remove_prefix_from_col(col):
    l = col.split('.')
    if (len(l) > 1):
        return f"{l[1]}"
    return l[0]
#########################################################

#########################################################
# transform list of obj into a dict mapped by key in obj
def transform_list_to_dict(objs, key):
    d = { obj[key]: obj for obj in objs }
    return { str(key): value for key, value in d.items() }
#########################################################

#########################################################
# rename all fields in collection #######################
def rename_all_fields_in_collection(collection):
    # Retrieve all documents from the collection
    documents = collection.find()

    # Update each document
    for doc in documents:
        # Get the document ID
        doc_id = doc["_id"]

        # Rename and remove fields
        updated_doc = rename_and_remove_fields(doc)

        # Remove old fields that contain periods
        update_operations = {
            "$set": updated_doc,
            "$unset": {key: "" for key in doc.keys() if '.' in key}
        }

        # Save the updated document back to the collection
        collection.update_one({"_id": doc_id}, update_operations)

    print("Fields containing periods renamed to camelCase or removed successfully.")
#########################################################

#########################################################
# remove all dot separated keys #########################
def remove_all_dot_separated_keys(collection):
    # Find all documents in the collection
    documents = collection.find()

    for document in documents:
        doc_id = document['_id']
        dot_keys = get_dot_separated_keys(document)

        if dot_keys:
            unset_query = {key: "" for key in dot_keys}
            # Remove the dot-separated keys from the document
            collection.update_one({'_id': doc_id}, {'$unset': unset_query})

    print("Dot-separated keys have been removed.")
#########################################################  

#########################################################
# update entire collection ##############################
def update_entire_collection(collection, update):
    result = collection.update_many({}, update)
    print(result)
# example: update_entire_collection(player_game_stats_historical_collection, { '$set': { 'season': 2023 }})
#########################################################

#########################################################
# get mongo pipeline to load player stats per season ####
def get_mongo_pipeline_for_player_stats_per_season(team_id, player_id, season, date_start, date_end):
    return [
        {
            "$match": {
                '$and': [
                    { 'dateStart': { '$gte': date_start } }, 
                    { 'dateStart': { '$lte': date_end } },
                ], 
                "$or": [
                    { "teamsHomeId": team_id },
                    { "teamsVisitorsId": team_id }
                ]
            }
        },
        {
            "$project": {
                "teamsHomePlayers": {
                    "$filter": {
                        "input": {"$objectToArray": "$teamsHomePlayers"},
                        "as": "player",
                        "cond": {"$eq": ["$$player.k", str(player_id) ]}
                    }
                },
                "teamsVisitorsPlayers": {
                    "$filter": {
                        "input": {"$objectToArray": "$teamsVisitorsPlayers"},
                        "as": "player",
                        "cond": {"$eq": ["$$player.k", str(player_id) ]}
                    }
                },
                "dateStart": 1
            }
        },
        {
            "$match": {
                "$or": [
                    {"teamsHomePlayers": {"$ne": []}},
                    {"teamsVisitorsPlayers": {"$ne": []}}
                ]
            }
        },
        {
            "$project": {
                "_id": 0,
                "playerStats": {
                    "$cond": {
                        "if": {"$gt": [{"$size": "$teamsHomePlayers"}, 0]},
                        "then": {"$arrayElemAt": ["$teamsHomePlayers.v", 0]},
                        "else": {"$arrayElemAt": ["$teamsVisitorsPlayers.v", 0]}
                    }
                },
                "dateStart": 1
            }
        }
    ]
#########################################################

#########################################################

In [17]:
#################### constants ##########################
#########################################################
nba_season_2023_start_date = '2023-10-24'

nba_season_2023_end_date = '2024-04-14'

cols_to_drop_for_game_stats = [
    "stage",
    "officials",
    "timesTied",
    "leadChanges",
    "nugget",
    "date.end",
    "date.duration",
    "status.clock",
    "status.halftime",
    "status.short",
    "status.long",
    "periods.current",
    "periods.total",
    "periods.endOfPeriod",
    "arena.name",
    "arena.city",
    "arena.state",
    "arena.country",
    "teams.home.logo",
    "scores.home.series.win",
    "scores.home.series.loss",
    "scores.home.win",
    "scores.home.loss",
    "teams.visitors.logo",
    "scores.visitors.series.win",
    "scores.visitors.series.loss",
    "scores.visitors.win",
    "scores.visitors.loss",
]

cols_to_drop_for_player_stats = [
    "comment",
    "team.nickname",
    "team.code",
    "team.name",
    "team.logo",
]

player_statistical_columns = [
    'playerStats.points',
    'playerStats.min', 
    'playerStats.fgm', 
    'playerStats.fga',
    'playerStats.fgp', 
    'playerStats.ftm', 
    'playerStats.fta',
    'playerStats.ftp', 
    'playerStats.tpm', 
    'playerStats.tpa',
    'playerStats.tpp', 
    'playerStats.offReb', 
    'playerStats.defReb',
    'playerStats.totReb', 
    'playerStats.assists', 
    'playerStats.pFouls',
    'playerStats.steals', 
    'playerStats.turnovers', 
    'playerStats.blocks',
    'playerStats.plusMinus'
]

#########################################################

In [18]:
############### nba_games_historical ####################
#########################################################

#########################################################
# load games data into nba_games_historical #######
def load_nba_games(season):    
    games_data_dict = []
    for team in get_team_nicknames_by_id():
        games_df = get_games_by_game_ids(season, teams.get(team))
        games_df['_id'] = games_df['id']
        games_data_dict.append(games_df.to_dict("records"))

    flat_games_data_dict = []
    for row in games_data_dict:
        flat_games_data_dict.extend(row)

    deduped_dict = {}
    for item in flat_games_data_dict:
        deduped_dict[item["_id"]] = item
    flat_games_data_dict = list(deduped_dict.values())
    
    game_data_to_insert = []
    for d in flat_games_data_dict:
        game_data_to_insert.append(rename_and_remove_fields(d))

    # Insert the data into the MongoDB collection
    result = nba_games_historical_collection.insert_many(game_data_to_insert)

    # Print the inserted IDs
    print("Inserted IDs:", result.inserted_ids)
#########################################################

#########################################################

In [19]:
######### player_game_stats_avgs_historical #############
#########################################################

#########################################################
# calculate player rolling stats averages ###############
def calculate_player_rolling_averages(df, window):
    # Calculate the rolling average for the selected columns
    try:
        numerical_cols = zero_non_numeric_values(df[player_statistical_columns])
        rolling_avgs = numerical_cols.rolling(window=window, min_periods=1).mean()
    except Exception as e:
        print(e)
        print(df)

    rolling_avgs = pd.concat([rolling_avgs, df[['playerStats.gameId', 'dateStart']]], axis=1)
    
    # Rename the columns to indicate they are rolling averages
    rolling_avgs = rolling_avgs.rename(columns=lambda x: remove_prefix_from_col(x))
    
    return rolling_avgs
#########################################################

#########################################################
# calculate player expanding stats averages #############
def calculate_player_expanding_averages(df):
    # Calculate the rolling average for the selected columns
    try:
        numerical_cols = zero_non_numeric_values(df[player_statistical_columns])
        expanding_avgs = numerical_cols.expanding().mean()
    except Exception as e:
        print(e)
        print(df)

    expanding_avgs = pd.concat([expanding_avgs, df[['playerStats.gameId', 'dateStart']]], axis=1)
    expanding_avgs = expanding_avgs.rename(columns=lambda x: remove_prefix_from_col(x))

    return expanding_avgs
#########################################################

#########################################################
# aggregate averages for player #########################
def aggregate_player_avgs_for_player(player_obj, team_id, season, start_date, end_date):
    player_doc = { 
        '_id': f"{player_obj['id']}_{team_id}_{season}", 
        'playerId': player_obj['id'],
        'teamId': team_id,
        'season': season,
        'firstname': player_obj['firstname'],
        'lastname': player_obj['lastname'],
        'birthday': player_obj['birth.date'],
        'countryOfBirth': player_obj['birth.country'],
    }
    
    pipeline = get_mongo_pipeline_for_player_stats_per_season(team_id, player_obj['id'], season, start_date, end_date)
    player_games = player_game_stats_historical_collection.aggregate(pipeline)
    
    normalized_df = pd.json_normalize(list(player_games)) 
    
    try:
        player_stats_numerical_cols = zero_non_numeric_values(normalized_df[player_statistical_columns])
        player_stats = pd.concat([player_stats_numerical_cols, normalized_df[['playerStats.gameId', 'dateStart']]], axis=1)
        player_stats = player_stats.rename(columns=lambda x: remove_prefix_from_col(x))
    except Exception as e:
        print(f"ERROR Parsing Player Data for: {player_obj['firstname']} {player_obj['lastname']}")
        return None
    
    expanding_avg_df = calculate_player_expanding_averages(normalized_df)
    rolling_avg_5_df = calculate_player_rolling_averages(normalized_df, 5)
    rolling_avg_10_df = calculate_player_rolling_averages(normalized_df, 10)
    
    player_doc['playerStats'] = transform_list_to_dict(player_stats.to_dict(orient='records'), 'gameId')
    player_doc['expandingAvg'] = transform_list_to_dict(expanding_avg_df.to_dict(orient='records'), 'gameId')
    player_doc['rollingAvg5'] = transform_list_to_dict(rolling_avg_5_df.to_dict(orient='records'), 'gameId')
    player_doc['rollingAvg10'] = transform_list_to_dict(rolling_avg_10_df.to_dict(orient='records'), 'gameId')

    return player_doc
#########################################################

#########################################################
# aggregate averages for team ###########################
def aggregate_player_avgs_per_team(team_id, season, start_date, end_date):
    player_data = get_player_per_team_and_season(team_id, season)
    all_players = []   
    for index, player in player_data.iterrows():
        player_avgs = aggregate_player_avgs_for_player(player, team_id, season, start_date, end_date)
        if player_avgs is not None:      
            all_players.append(player_avgs)
    return all_players
#########################################################

#########################################################
# load player avgs per game #############################
def load_player_avgs_through_season(season, start_date, end_date, avgs_tag):
    teams = get_team_nicknames_by_id()
    for team_nickname, team_id in teams.items():
        all_players_on_team_avgs = aggregate_player_avgs_per_team(team_id, season, start_date, end_date)
        player_game_stats_avgs_historical_collection.insert_many(all_players_on_team_avgs)
#########################################################

#########################################################

In [20]:
########### player_game_stats_historical ################
#########################################################

#########################################################
# get game ids for season ###############################
def get_game_for_season(season):
    return collection.find({ "season": season })
#########################################################

#########################################################
# load player game data into player_game_stats_historical
def load_player_game_stats(season):
    games = get_game_for_season(season)
    all_player_stats_per_game = []
    for game in games:
        game_id = game['_id']
        home_team_id = game['teamsHomeId']
        visitors_team_id = game['teamsVisitorsId']

        players_dict_list = get_players_per_game_df(game_id).to_dict("records")

        # Initialize a defaultdict
        players_grouped_by_team = defaultdict(list)
        # Group the data
        for item in players_dict_list:
            players_grouped_by_team[item['team.id']].append(item)
        players_grouped_by_team['teamsHomePlayers'] = players_grouped_by_team.pop(home_team_id)
        players_grouped_by_team['teamsHomePlayers'] = transform_list_to_dict(players_grouped_by_team['teamsHomePlayers'], 'player.id')
        players_grouped_by_team['teamsVisitorsPlayers'] = players_grouped_by_team.pop(visitors_team_id)
        players_grouped_by_team['teamsVisitorsPlayers'] = transform_list_to_dict(players_grouped_by_team['teamsVisitorsPlayers'], 'player.id')
        players_grouped_by_team['teamsHomeId'] = home_team_id
        players_grouped_by_team['teamsVisitorsId'] = visitors_team_id
        players_grouped_by_team['season'] = season
        players_grouped_by_team['dateStart'] = game['dateStart']
        players_grouped_by_team['_id'] = game_id
        all_player_stats_per_game.append(rename_and_remove_fields(players_grouped_by_team))
    
    result = player_game_stats_historical_collection.insert_many(all_player_stats_per_game)
    
    # Print the inserted IDs
    print("Inserted IDs:", result.inserted_ids)
#########################################################

#########################################################

In [21]:
#########################################################
# init_nba_games_historical_loader ######################
def init_nba_games_historical_loader(season):
    load_nba_games(season)
#########################################################

In [22]:
#########################################################
# init_player_game_stats_avgs_historical ################
def init_player_game_stats_avgs_historical(season, start_date, end_date, avgs_tag):
    load_player_avgs_through_season(season, start_date, end_date, avgs_tag)
#########################################################

In [23]:
#########################################################
# init_player_game_stats_historical #####################
def init_load_player_game_stats(season):
    load_player_game_stats(season)
#########################################################

In [24]:
init_player_game_stats_avgs_historical("2023", nba_season_2023_start_date, nba_season_2023_end_date, "poop")

/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be se

ERROR Parsing Player Data for: Miles Norris


/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be se

ERROR Parsing Player Data for: Wenyen Gabriel


/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be se

...

/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be se

ERROR Parsing Player Data for: Makur Maker


/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be se

ERROR Parsing Player Data for: Taj Gibson
ERROR Parsing Player Data for: Xavier Cooks


/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
/var/folders/s8/h_031z5d7mbf8ckkr3tnb9rw0000gn/T/ipykernel_80939/3141938464.py:68: SettingWithCopyWarning: 
A value is trying to be se